# Pareto smoothed importance sampling leave-one-out (PSIS LOO)

We often need to evaluate the performance of the Bayesian models on a test data or out of sample data after training (MCMC) for model comparison or selection. Crossvalidation (CV) is one of the effective methods for estimating the out of sample predictive accuracy.

### LOO CV 
Exact leave one out cross validation (LOO CV) requires performing MCMC on several training datasets and testing on the one left out datapoint each time. The computation can be very time consuming (from a few days to several months), depending on the size of the size of the dataset. However, an approximation of leave one out cross validation (LOO CV) can be estimated using importance sampling within minutes.

Consider data points $y_1,\ldots,y_n$ which are independent given parameters $\theta$;

Likelihood: $p(y|\theta)=\prod_{i=1}^{n} p(y_i|\theta)$

Assume prior: $p(\theta)$

Yielding posterior: $p(\theta|y)$

The Bayesian LOO CV estimate is represented using (elpd) expected log pointwise predictive desnsity:

$$
\text{elpd}_{\text{loo-cv}}
=
\sum_{i=1}^{n}
\log p(y_i|y_{-i}),
$$
where

$$
p(y_i|y_{-i})
=
\int p(y_i|\theta)p(\theta|y_{-i})d\theta
$$

is the leave-one-out predictive density given the data without the ith data point.

From [Gelfand, Dey, and Chang (1992)](https://statistics.stanford.edu/technical-reports/model-determination-using-predictive-distributions-implementation-sampling-based), the exact loo estimate; $p(y_i|y_{-i})$ can be approximated by importance sampling.  The approximation is obtained using posteior draws $\theta^s$ from the full posterior distribution $p(\theta|y)$ and corresponding importance ratios.

### Importance ratios
Assume the number of posterior samples is *S*. Importance ratio for observation *i* with posterior sample index *s* is given by: 
$$
r_i^s
=
\frac{1}{p(y_i|\theta^s)}
\propto
\frac{p(\theta^s|y_{-i})}{p(\theta^s|y)}
$$
  
Importance sampling leave-one-out (IS-LOO) predictive distribution:

$$
p(\tilde{y}_i|y_{-i})
\approx
\frac{\sum_{s=1}^{S} r_i^s p(\tilde{y}_i|\theta^s)}
{\sum_{s=1}^{S} r_i^s}.
$$

Evaluating at the held-out data point $y_i$, we get

$$
p(y_i|y_{-i})
\approx
\frac{1}
{\frac{1}{S}\sum_{s=1}^{S}\frac{1}{p(y_i|\theta^s)}}.
$$

The problem with IS-LOO is that the posterior $p(\theta|y)$ is likely to have a smaller variance and thinner tails than $p(\theta|y_{-i})$. Thus, a direct use of the above expression for $p(y_i|y_{-i})$ induces instability because the importance ratios/weights can have large or infinite variance.

### Pareto smoothed importance sampling (PSIS)
The distribution of the importance weights used in LOO may have a long right tail, so direct sampling can lead to one or a few very large weights. PSIS applies a smoothing procedure to the importance weights by fitting a Pareto distribution to the upper tail of the importance ratios ($r_i^s$). The largest weights in $r_i^s$ are then replaced by expected quantiles $w_i^s$ from the fitted Pareto distribution, which are more well behaved than the original $r_i^s$ from which they are constructed.

#### PSIS LOO CV

PSIS estimate of the LOO expected log pointwise predictive density is given by:
$$
\text{elpd}_{\text{psis-loo-cv}}
=
\sum_{i=1}^{n}
\log
\left(
\frac{
\sum_{s=1}^{S} w_i^s \, p(y_i \mid \theta^s)
}{
\sum_{s=1}^{S} w_i^s
}
\right).
$$

The shape parameter *k* of the modeled Pareto distribution can be used to assess the reliability of the estimate:

| Pareto shape parameter | Interpretation |
|---|---|
| *k* $< 1 - \frac{1}{\log_{10}(S)}$ | The PSIS estimate is expected to be accurate. |
| *k* $< \min\left(1 - \frac{1}{\log_{10}(S)},\, 0.7\right)$ | The PSIS estimate is expected to be reliable. |
| $0.7 < $ *k* $ < 1$ | It becomes computationally expensive to obtain an accurate estimate. |
| *k* $> 1$ | Mean and variance of the importance weights does not exist, and PSIS estimates become invalid. |
### PSIS LOO PIT 
The ordinary leave-one-out (LOO) probability integral transform (PIT) value for observation $y_i$ is

$$
\text{PIT}_i
=
P(\tilde y_i \le y_i \mid y_{-i})
=
\int
P(\tilde y_i \le y_i \mid \theta)
\, p(\theta \mid y_{-i})
\, d\theta.
$$

Using importance sampling:

$$
\text{PIT}_i
\approx
\frac{
\sum_{s=1}^{S}
r_i^s \,
F(y_i \mid \theta^s)
}{
\sum_{s=1}^{S} r_i^s
},
$$

where

$$
F(y_i \mid \theta^s)
=
P(\tilde y_i \le y_i \mid \theta^s)
$$

is the posterior predictive cumulative distribution function (CDF) evaluated at the observed value $y_i$.

In Pareto smoothed importance sampling leave-one-out (PSIS-LOO), the unstable importance ratios $r_i^s$ are replaced by Pareto-smoothed weights $w_i^s$:

$$
\text{PSIS-LOO-PIT}_i
\approx
\frac{
\sum_{s=1}^{S}
w_i^s \,
F(y_i \mid \theta^s)
}{
\sum_{s=1}^{S} w_i^s
}.
$$

### When to use
PSIS-LOO-CV is primarily used for model comparison and model selection. When we have two or more candidate models, elpd values can be used to evaluate which model is expected to generalize better. A higher elpd value indicates that a model is expected to predict unseen data more accurately. 

PSIS-LOO-PIT calculates the probability integral transform (CDF) and is used to evaluate the reliability of the model. If the PIT values $\text{PSIS-LOO-PIT}_i$ follow a uniform distribution, then the Bayesian model is considered reliable.

Both metrics can also be used to detect outliers. In PSIS-LOO-CV, if a data point has Pareto *k* $> 0.7$ or a highly negative $\text{elpd}_{i}$ value, then it is a potential outlier. Similarly, in PSIS-LOO-PIT, if a data point has PIT values $\text{PSIS-LOO-PIT}_i$ close to 0 or 1, then it lies in the long tail of the predictive distribution, indicating that the point may be an outlier.



Overall, PSIS-LOO provides an efficient and fully Bayesian approach for estimating out-of-sample predictive performance and assessing model reliability. Due to its stability and diagnostics, it is widely preferred in modern PPL over older criteria such as DIC.

### References

1. Gelfand & Chang (1992). *Model determination using predictive distributions with implementation via sampling-based methods*.
https://statistics.stanford.edu/technical-reports/model-determination-using-predictive-distributions-implementation-sampling-based

2. Vehtari, Gelman, & Gabry (2017). *Practical Bayesian model evaluation using leave-one-out cross-validation and WAIC*.  
https://arxiv.org/abs/1507.04544

3. Vehtari, Simpson, Gelman, Yao, & Gabry (2024). *Pareto smoothed importance sampling*.  
https://arxiv.org/abs/1507.02646 